# Exercise 1: Webscraping

Author: Georg Ahnert

In this exercise, we will first recap some basic Pandas functionality for handeling datasets in Python. Then, we will have a look into webscraping and crawling.

You will first have to install the following dependencies:

In [ ]:
%pip install pandas beautifulsoup4 requests urllib3 --upgrade

## Part 1: Pandas Recap

First, let's recap some basic Pandas functionalities. 

In [2]:
import pandas as pd

/Users/georg/anaconda3/envs/webscraping_course/lib/python3.12/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/Users/georg/anaconda3/envs/webscraping_course/lib/python3.12/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


### Create a Pandas DataFrame

Call the variable `test_df`. It should have the following structure:

|   | column_A | column_B |
|---|----------|----------|
| 2 | 2 | 3  |
| 3 | 4 | 15 |
| 4 | 6 | 75 |

In [3]:
# There are multiple ways in which a DataFrame can be initialized, this is one
test_df = pd.DataFrame([
    {'column_A': 2, 'column_B': 3},
    {'column_A': 4, 'column_B': 15},
    {'column_A': 6, 'column_B': 75},
], index=[2,3,4]) # don't forget adjusting the index if needed
test_df

,column_A,column_B
2,2,3
3,4,15
4,6,75


Next, we will work with the Quality of Government Dataset and perform some basic operations

In [4]:
# read the data directly online:
df=pd.read_csv("http://www.qogdata.pol.gu.se/data/qog_std_cs_jan22.csv")
df.head()

,ccode,cname,ccode_qog,cname_qog,ccodealp,ccodecow,version,aii_acc,aii_aio,aii_cilser,...,wwbi_sprpemps,wwbi_sprpempt,wwbi_spupempn,wwbi_spupempp,wwbi_spupemps,wwbi_spupempt,wwbi_tertiarypubsec,yri_yoe,yri_yri35,yri_yri40
0,4,Afghanistan,4,Afghanistan,AFG,700.0,QoGStdCSjan22,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,8,Albania,8,Albania,ALB,339.0,QoGStdCSjan22,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2013.0,26.869801,0.565712
2,12,Algeria,12,Algeria,DZA,615.0,QoGStdCSjan22,6.25,12.5,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,20,Andorra,20,Andorra,AND,232.0,QoGStdCSjan22,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2015.0,120.760000,0.863975
4,24,Angola,24,Angola,AGO,540.0,QoGStdCSjan22,18.75,17.5,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### How many rows does this dataset have?

In [5]:
len(df)  # the 'size' of the DataFrame is equal to it's number of rows

194

### How many columns does this dataset have?

In [6]:
len(df.columns)  # df.columns returns a list of all columns

1714

### For the following tasks, select these columns from the dataset: 

"cname", "wdi_pop", "wdi_popgr", "wdi_gdpcapcur", "wdi_gdpcapgr", "wdi_area", "wdi_broadb", "ht_region"

In [7]:
# use a list of column names to select multiple columns
df2 = df[["cname", "wdi_pop", "wdi_popgr", "wdi_gdpcapcur", "wdi_gdpcapgr", "wdi_area", "wdi_broadb", "ht_region"]]
df2.head()

,cname,wdi_pop,wdi_popgr,wdi_gdpcapcur,wdi_gdpcapgr,wdi_area,wdi_broadb,ht_region
0,Afghanistan,37171920.0,2.384309,493.756592,-1.194900,652860.0,0.043041,8
1,Albania,2866376.0,-0.246732,5284.380371,4.328395,27400.0,12.555659,1
2,Algeria,42228416.0,2.007399,4153.956055,-0.811233,2381741.0,7.262936,3
3,Andorra,77008.0,0.014285,41791.968750,1.574254,470.0,46.311977,5
4,Angola,30809788.0,3.276145,3289.644043,-5.162112,1246700.0,0.355605,4


### Rename these columns to: "country", "population","population_growth", "gdp_per_capita", "gdp_growth", "area", "internet", "region"

In [8]:
# pass a dictionary to rename() to rename the columns
df2 = df2.rename(columns={"cname" : "name",
            "wdi_pop" : "population",
            "wdi_popgr" : "population_growth",
            "wdi_gdpcapcur" : "gdp_per_capita",
            "wdi_gdpcapgr" : "gdp_growth",
            "wdi_area" : "area",
            "wdi_broadb" : "internet",
            "ht_region" : "region"
           })
df2.head()

,name,population,population_growth,gdp_per_capita,gdp_growth,area,internet,region
0,Afghanistan,37171920.0,2.384309,493.756592,-1.194900,652860.0,0.043041,8
1,Albania,2866376.0,-0.246732,5284.380371,4.328395,27400.0,12.555659,1
2,Algeria,42228416.0,2.007399,4153.956055,-0.811233,2381741.0,7.262936,3
3,Andorra,77008.0,0.014285,41791.968750,1.574254,470.0,46.311977,5
4,Angola,30809788.0,3.276145,3289.644043,-5.162112,1246700.0,0.355605,4


### Create a _categorical_ column from the region codes
the codes (in order correspond to the regions as follows):
"Eastern Europe", "Latin America", "North Africa & Middle East", "Sub-Saharan Africa", "Western Europe and North America", "East Asia","South-East Asia", "South Asia", "Pacific", "Caribbean"

In [9]:
# replace() can be used to replace the values based on the keys and values of a dictionary

regions = {
    1: "Eastern Europe",
    2: "Latin America",
    3: "North Africa & Middle East",
    4: "Sub-Saharan Africa",
    5: "Western Europe and North America",
    6: "East Asia",
    7: "South-East Asia",
    8: "South Asia",
    9: "Pacific",
    10: "Caribbean",
}

df2['region'] = df2['region'].replace(regions).astype('category')
df2.head()

,name,population,population_growth,gdp_per_capita,gdp_growth,area,internet,region
0,Afghanistan,37171920.0,2.384309,493.756592,-1.194900,652860.0,0.043041,South Asia
1,Albania,2866376.0,-0.246732,5284.380371,4.328395,27400.0,12.555659,Eastern Europe
2,Algeria,42228416.0,2.007399,4153.956055,-0.811233,2381741.0,7.262936,North Africa & Middle East
3,Andorra,77008.0,0.014285,41791.968750,1.574254,470.0,46.311977,Western Europe and North America
4,Angola,30809788.0,3.276145,3289.644043,-5.162112,1246700.0,0.355605,Sub-Saharan Africa


In [10]:
df2.info()

<class 'pandas.DataFrame'>
RangeIndex: 194 entries, 0 to 193
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype   
---  ------             --------------  -----   
 0   name               194 non-null    str     
 1   population         192 non-null    float64 
 2   population_growth  192 non-null    float64 
 3   gdp_per_capita     189 non-null    float64 
 4   gdp_growth         188 non-null    float64 
 5   area               193 non-null    float64 
 6   internet           189 non-null    float64 
 7   region             194 non-null    category
dtypes: category(1), float64(6), str(1)
memory usage: 11.0 KB


### Select the five countries with the highest population

In [11]:
# use sort_values to sort by a column
df2.sort_values("population", ascending=False).head(5)

,name,population,population_growth,gdp_per_capita,gdp_growth,area,internet,region
36,China,1.392730e+09,0.455900,9976.676758,6.264210,9424703.0,28.535189,East Asia
76,India,1.352642e+09,1.037828,1996.915039,5.433077,2973190.0,1.343297,South Asia
186,United States of America (the),3.268382e+08,0.526435,63064.417969,2.455679,9147420.0,33.860367,Western Europe and North America
77,Indonesia,2.676705e+08,1.134507,3893.859619,3.987825,1877519.0,3.315313,South-East Asia
131,Pakistan,2.122283e+08,2.057546,1482.213013,3.681035,770880.0,0.987346,South Asia


### What are the mean values for each attribute?

In [12]:
# use describe() for general statistics
df2.describe()

,population,population_growth,gdp_per_capita,gdp_growth,area,internet
count,1.920000e+02,192.000000,189.000000,188.000000,1.930000e+02,189.000000
mean,3.930178e+07,1.295607,16210.073792,1.856232,6.707763e+05,13.919198
std,1.447560e+08,1.145389,26298.824404,2.982412,1.841246e+06,14.540677
min,1.067800e+04,-1.768331,271.752502,-12.131799,2.027000e+00,0.001822
25%,2.037214e+06,0.473990,2014.570312,0.585977,2.467000e+04,0.871731
50%,8.861660e+06,1.231601,6145.818848,1.919880,1.203400e+05,8.551120
75%,2.859579e+07,2.060348,17745.255859,3.823976,5.279700e+05,25.941820
max,1.392730e+09,4.921024,185978.609375,13.446087,1.637687e+07,55.818920


In [13]:
df2.describe().loc['mean']  # only get the 'mean' row from the DataFrame that describe() produces

population           3.930178e+07
population_growth    1.295607e+00
gdp_per_capita       1.621007e+04
gdp_growth           1.856232e+00
area                 6.707763e+05
internet             1.391920e+01
Name: mean, dtype: float64

### Which country has the highest population in the region "South-East Asia"?

In [14]:
# select the South-East Asia region
# then, sort rows by population
# finally, only show the country with highest population
df2[df2.region == "South-East Asia"].sort_values("population", ascending=False).head(1)

,name,population,population_growth,gdp_per_capita,gdp_growth,area,internet,region
77,Indonesia,267670544.0,1.134507,3893.859619,3.987825,1877519.0,3.315313,South-East Asia


### Create a new column "population_density"

In [15]:
df2["population_density"] = df2['population'] / df2['area']  # calculate directly from two other columns/Series
df2.head()

,name,population,population_growth,gdp_per_capita,gdp_growth,area,internet,region,population_density
0,Afghanistan,37171920.0,2.384309,493.756592,-1.194900,652860.0,0.043041,South Asia,56.937046
1,Albania,2866376.0,-0.246732,5284.380371,4.328395,27400.0,12.555659,Eastern Europe,104.612263
2,Algeria,42228416.0,2.007399,4153.956055,-0.811233,2381741.0,7.262936,North Africa & Middle East,17.730062
3,Andorra,77008.0,0.014285,41791.968750,1.574254,470.0,46.311977,Western Europe and North America,163.846809
4,Angola,30809788.0,3.276145,3289.644043,-5.162112,1246700.0,0.355605,Sub-Saharan Africa,24.713073


## Part 2: Web Scraping

### Intro to HTML
HTML - Hyper Text Markup Language (see also [Wikipedia](https://en.wikipedia.org/wiki/HTML))

HTML elements are defined by tags
```
<b>Bold text</b>
```
Tags have attributes
```
<span class="uni">University of Mannheim</span>
```
Tags could be nested
```
<div id="uni-list" class="sfsdf" attribute1="xasdas" attribute2="asfasd">
 <span class="uni">University of Mannheim</span>
 <span class="uni">RWTH Aachen</span>
 <a class="uni">Hello</a>
</div>
```


Here's a [web page example](https://www.uni-mannheim.de/en/academics/programs/). Here's a video that recaps the most important elements of the language: [YouTube](https://youtu.be/salY_Sm6mv4).

### Querying HTML pages with BeautifulSoup

HTML-elements could be selected by name
```
a
```
By ID
```
#uni-list
```
By class
```
.uni
```
Nested selection
```
#uni-list .uni
```

```
#uni-list span.uni
```

See this nice [introduction to CSS selectors](https://developer.mozilla.org/en-US/docs/Web/CSS/CSS_selectors/Selectors_and_combinators).
See also the [BeautifulSoup Documentation](https://beautiful-soup-4.readthedocs.io/en/latest/)

In [16]:
from bs4 import BeautifulSoup  # import the BeautifulSoup class from the bs4 package

In [17]:
# an example HTML document in the form of a string
html_doc = '<html lang="en"><head><title>Test document</title></head><body><div id="uni-list"><div class="sub"><a href="https://www.uni-mannheim.de/" class="uni">University of Mannheim</a></div><div class="sub"><a href="http://rwth-aachen.de" class="uni">RWTH Aachen</a></div></div></body></html>'

# initialize a BeautifulSoup object with the given HTML
soup = BeautifulSoup(html_doc)
type(soup)

bs4.BeautifulSoup

In [18]:
# we can also specify the type of parser we want to use
# they work differently well with non-conforming HTML
soup = BeautifulSoup(html_doc, parser='html.parser')

In [19]:
soup  # let's see what we got

<html lang="en"><head><title>Test document</title></head><body><div id="uni-list"><div class="sub"><a class="uni" href="https://www.uni-mannheim.de/">University of Mannheim</a></div><div class="sub"><a class="uni" href="http://rwth-aachen.de">RWTH Aachen</a></div></div></body></html>

In [20]:
print(soup.prettify())  # add indentation

<html lang="en">
 <head>
  <title>
   Test document
  </title>
 </head>
 <body>
  <div id="uni-list">
   <div class="sub">
    <a class="uni" href="https://www.uni-mannheim.de/">
     University of Mannheim
    </a>
   </div>
   <div class="sub">
    <a class="uni" href="http://rwth-aachen.de">
     RWTH Aachen
    </a>
   </div>
  </div>
 </body>
</html>



### Accessing Elements in the Soup

In [21]:
soup.title  # access the first 'title' element in the soup

<title>Test document</title>

In [22]:
soup.title.text  # get the text inside the first 'title' element

'Test document'

In [23]:
print(soup.div.prettify())  # access the first 'div' element and print it pretty

<div id="uni-list">
 <div class="sub">
  <a class="uni" href="https://www.uni-mannheim.de/">
   University of Mannheim
  </a>
 </div>
 <div class="sub">
  <a class="uni" href="http://rwth-aachen.de">
   RWTH Aachen
  </a>
 </div>
</div>



In [24]:
soup.a  # access the first anchor element (hyperlink)

<a class="uni" href="https://www.uni-mannheim.de/">University of Mannheim</a>

But what if we want to get **all** hyperlinks? We can use **CSS selectors** to achieve this.

In [25]:
result = soup.select('a')  # returns a list of all elements that match the name 'a'
result

[<a class="uni" href="https://www.uni-mannheim.de/">University of Mannheim</a>,
 <a class="uni" href="http://rwth-aachen.de">RWTH Aachen</a>]

In [26]:
soup.select('span')  # the list can also be empty or have only one element

[]

In [27]:
second = soup.select('a')[1]  # we can extract entries from the list just as usual
second

<a class="uni" href="http://rwth-aachen.de">RWTH Aachen</a>

### More advanced selectors

### Select all elements that have the ID 'uni-list'

In [28]:
result = soup.select('#uni-list')
len(result)

1

In [29]:
print(result[0].prettify()) # print the element we found in a pretty way

<div id="uni-list">
 <div class="sub">
  <a class="uni" href="https://www.uni-mannheim.de/">
   University of Mannheim
  </a>
 </div>
 <div class="sub">
  <a class="uni" href="http://rwth-aachen.de">
   RWTH Aachen
  </a>
 </div>
</div>



### Select all elements that are of class 'uni'

In [30]:
soup.select('.uni')

[<a class="uni" href="https://www.uni-mannheim.de/">University of Mannheim</a>,
 <a class="uni" href="http://rwth-aachen.de">RWTH Aachen</a>]

### Select all elements that have name 'div' and class 'sub'

In [31]:
soup.select('div.sub')

[<div class="sub"><a class="uni" href="https://www.uni-mannheim.de/">University of Mannheim</a></div>,
 <div class="sub"><a class="uni" href="http://rwth-aachen.de">RWTH Aachen</a></div>]

### Select all 'div' elements that are descendents of an element with id 'uni-list'

In [32]:
soup.select('#uni-list div')

[<div class="sub"><a class="uni" href="https://www.uni-mannheim.de/">University of Mannheim</a></div>,
 <div class="sub"><a class="uni" href="http://rwth-aachen.de">RWTH Aachen</a></div>]

In [33]:
soup.select('div')  # in contrast to the previous query, this also returns the parent-div as well

[<div id="uni-list"><div class="sub"><a class="uni" href="https://www.uni-mannheim.de/">University of Mannheim</a></div><div class="sub"><a class="uni" href="http://rwth-aachen.de">RWTH Aachen</a></div></div>,
 <div class="sub"><a class="uni" href="https://www.uni-mannheim.de/">University of Mannheim</a></div>,
 <div class="sub"><a class="uni" href="http://rwth-aachen.de">RWTH Aachen</a></div>]

### Loading actual web data

In [34]:
import requests

url = "https://www.uni-mannheim.de/en/academics/before-your-studies/programs/"

page = requests.get(url).text  # get the content at that URL and store the page source

soup = BeautifulSoup(page)  # initialize beautiful soup

In [ ]:
soup  # let's see what we got

Let's try to extract the course programs that Uni Mannheim offers

In [36]:
courses = soup.select('.uma-ps-result-item')
len(courses)

77

In [ ]:
print(courses[0].prettify())  # inspect the first element

In [ ]:
result = soup.select('.uma-ps-result-item-title')  # we're only interested in the programs' names
result

In [39]:
[x.text for x in result]  # only the text, not the tags

["Bachelor's Program in Business Administration",
 "Bachelor's Program in Current English Linguistics and Literary Studies",
 "Bachelor's Program in German Studies: Language, Literature, Media",
 "Bachelor's Program in History",
 "Bachelor's Program in Culture and Economy",
 "Bachelor's Program in Culture and Economy: English and American Studies",
 "Bachelor's Program in Culture and Economy: German Studies",
 "Bachelor's Program in Culture and Economy: History",
 "Bachelor's Program in Culture and Economy: Media and Communication Studies",
 "Bachelor's Program in Culture and Economy: Philosophy",
 "Bachelor's Program in Culture and Economy: French Studies",
 "Bachelor's Program in Culture and Economy: Italian Studies",
 "Bachelor's Program in Culture and Economy: Spanish Studies",
 'Bachelor Lehramt Gymnasium: Bildende Kunst',
 'Bachelor Lehramt Gymnasium: Musik',
 "Bachelor's Program in Media and Communication Studies",
 "Bachelor's Program in Teacher Education",
 "Bachelor's Program

### Web Crawling

Another thing we could be interested in is all the testimonials that are published on the course programs' websites:

In [40]:
result = soup.select('.uma-ps-results a')  # get all the anchor elements

for element in result:
    print(element['href'])  # extract the hyperlinks

/en/academics/before-your-studies/programs/bsc-business-administration/
/en/academics/before-your-studies/programs/ba-cells/
/en/academics/before-your-studies/programs/ba-german-studies/
/en/academics/before-your-studies/programs/b-a-history/
/en/academics/before-your-studies/programs/bachelors-program-in-culture-and-economy/
/en/bakuwi-english-and-american-studies/
/en/academics/before-your-studies/programs/bakuwi-german-studies/
/en/academics/before-your-studies/programs/bakuwi-history/
/en/academics/before-your-studies/programs/bakuwi-media-and-communication-studies/
/en/academics/before-your-studies/programs/bakuwi-philosophy/
/en/academics/before-your-studies/programs/bakuwi-french-studies/
/en/academics/before-your-studies/programs/bakuwi-italian-studies/
/en/academics/before-your-studies/programs/bakuwi-spanish-studies/
/en/academics/before-your-studies/programs/bachelor-lehramt-gymnasium-kunst/
/en/academics/before-your-studies/programs/bachelors-program-in-media-and-communicat

In [ ]:
# follow all the links
for element in result:
    url = element['href']

    # convert relative to absolute links
    if not(url.startswith('https:/')):
        url = 'https://www.uni-mannheim.de' + url
    
    # get the linked page
    page = requests.get(url).text
    soup = BeautifulSoup(page, 'html.parser')
    testimonial_tags = soup.select('.testimonial-quote')
    
    # if there's a testimonial, print the first one
    if len(testimonial_tags) > 0:
        testimonial = testimonial_tags[0]
        print(testimonial.text.strip())